In [7]:
import sys
print(sys.executable)

/home/nineleaps/PythonSession/.venv/bin/python


In [9]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu numpy

  Using cached faiss_cpu-1.15.0-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
Using cached faiss_cpu-1.15.0-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.8 MB)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/nineleaps/PythonSession/.venv/bin/python -m pip install --upgrade pip


In [10]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
TOP_K = 3

KNOWLEDGE_BASE = [
    "To reset your password, click on 'Forgot Password' on the login page and follow the emailed instructions.",
    "You can update your billing address from the Account Settings > Billing section.",
    "If your card payment fails, please check with your bank or try a different payment method.",
    "To close or deactivate your account, go to Settings > Account > Delete Account.",
    "Login issues are often caused by an expired session; try clearing your browser cache and logging in again.",
    "You can view and download past invoices from the Billing History page.",
    "Two-factor authentication can be enabled from Security Settings for extra account protection.",
    "If you're locked out of your account after multiple failed login attempts, wait 15 minutes before retrying.",
    "To change your registered email address, verify your new email via the confirmation link we send you.",
    "Subscription plans can be upgraded or downgraded anytime from the Billing > Plans page.",
    "Refunds are processed within 5-7 business days back to the original payment method.",
    "If you forgot your username, use the 'Forgot Username' link and enter your registered email.",
]

print(f"Loading embedding model: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)

embeddings = model.encode(KNOWLEDGE_BASE, convert_to_numpy=True).astype("float32")
print(f"Embedding matrix shape: {embeddings.shape}")

/home/nineleaps/PythonSession/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 4200.50it/s]


Embedding matrix shape: (12, 384)


In [11]:
# Normalise embeddings so L2 distance behaves like cosine similarity
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings)

print(f"Total vectors stored in FAISS index: {index.ntotal}")

Total vectors stored in FAISS index: 12


In [12]:
def search(query: str, k: int = TOP_K):
    """Embed a query, normalise it, and return the top-k matches."""
    query_vec = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)

    distances, indices = index.search(query_vec, k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        cosine_sim = 1 - (dist / 2)  # convert normalised-L2 distance back to cosine similarity
        results.append((rank, cosine_sim, KNOWLEDGE_BASE[idx]))
    return results


def print_results(results):
    print(f"{'Rank':<6}{'Score':<10}Matched Sentence")
    print("-" * 80)
    for rank, score, sentence in results:
        print(f"{rank:<6}{score:<10.4f}{sentence}")
    print()

In [13]:
sample_queries = [
    "I forgot my password, how do I get back in?",
    "How do I get a refund for my subscription?",
    "My account got locked after too many login tries.",
]

for q in sample_queries:
    print(f"Query: {q}")
    print_results(search(q))

Query: I forgot my password, how do I get back in?
Rank  Score     Matched Sentence
--------------------------------------------------------------------------------
1     0.7228    To reset your password, click on 'Forgot Password' on the login page and follow the emailed instructions.
2     0.5742    If you forgot your username, use the 'Forgot Username' link and enter your registered email.
3     0.5375    If you're locked out of your account after multiple failed login attempts, wait 15 minutes before retrying.

Query: How do I get a refund for my subscription?
Rank  Score     Matched Sentence
--------------------------------------------------------------------------------
1     0.4705    Refunds are processed within 5-7 business days back to the original payment method.
2     0.4304    Subscription plans can be upgraded or downgraded anytime from the Billing > Plans page.
3     0.4186    You can update your billing address from the Account Settings > Billing section.

Query: My acc

In [14]:
my_query = "How do I turn on two-factor authentication?"  # <-- change this and re-run

print(f"Query: {my_query}")
print_results(search(my_query))

Query: How do I turn on two-factor authentication?
Rank  Score     Matched Sentence
--------------------------------------------------------------------------------
1     0.7969    Two-factor authentication can be enabled from Security Settings for extra account protection.
2     0.3274    To reset your password, click on 'Forgot Password' on the login page and follow the emailed instructions.
3     0.2793    To close or deactivate your account, go to Settings > Account > Delete Account.



In [15]:
while True:
    query = input("Query> ").strip()
    if query.lower() == "exit":
        print("Goodbye!")
        break
    if not query:
        continue
    print_results(search(query))

Query>  hi


Rank  Score     Matched Sentence
--------------------------------------------------------------------------------
1     0.1439    If you forgot your username, use the 'Forgot Username' link and enter your registered email.
2     0.1276    To close or deactivate your account, go to Settings > Account > Delete Account.
3     0.0816    To change your registered email address, verify your new email via the confirmation link we send you.



Query>  change email


Rank  Score     Matched Sentence
--------------------------------------------------------------------------------
1     0.7186    To change your registered email address, verify your new email via the confirmation link we send you.
2     0.4605    If you forgot your username, use the 'Forgot Username' link and enter your registered email.
3     0.4098    To reset your password, click on 'Forgot Password' on the login page and follow the emailed instructions.



Query>  exit


Goodbye!
